# Week 2 Assignment — Machine Learning for Forecasting
### Multi-Agent Forecasting Project

**Name:** Student  
**Date:** 2026-06-19  

---

This assignment builds directly on Week 1. You already know how to explore and decompose a time series — now you'll learn how to *forecast* one using machine learning models. By the end, you'll have built and evaluated multiple forecasting approaches on a real-world dataset.

**Topics covered:**
1. Linear Regression with Time Features (Trend Modelling)
2. Modelling Seasonality with Fourier Features
3. Lag Features & Serial Dependence
4. Decision Trees, Random Forests, and XGBoost for Time Series
5. Hybrid Models: Linear + XGBoost
6. Putting It All Together — Multi-Step Forecasting

> **Dataset:** We'll use the **Melbourne Daily Minimum Temperatures** dataset (1981–1990) for Sections 1–5 — it has a clear annual seasonal pattern and manageable size. Section 6 returns to the **Air Passengers** dataset from Week 1.

In [ ]:
# Run this cell first — installs/imports everything you need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from statsmodels.graphics.tsaplots import plot_pacf
from xgboost import XGBRegressor

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Load dataset ──────────────────────────────────────────────────────────────
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
df = pd.read_csv(url, parse_dates=['Date'], index_col='Date')
df.columns = ['temp']

print("Dataset shape:", df.shape)
print(df.head())

---
## Section 1 — Linear Regression with Time Features (Trend Modelling)

The simplest way to model a trend is to use **time itself as a feature**. We create a numeric counter `t = 0, 1, 2, ...` for each time step and fit a linear model:

$$\hat{y}_t = w_0 + w_1 \cdot t$$

For a curved trend, we extend this to a **polynomial** by adding $t^2$, $t^3$, etc.

`DeterministicProcess` from `statsmodels` makes this easy — `order=1` gives a linear trend, `order=2` gives quadratic, and so on.

In [ ]:
# --- EXAMPLE: Fitting a linear trend with DeterministicProcess ---

dp_example = DeterministicProcess(
    index=df.index,
    constant=True,
    order=1,
    drop=True,
)

X_example = dp_example.in_sample()
print("Feature matrix shape:", X_example.shape)
print(X_example.head())

### Exercise 1.1 — Fit a Linear Trend

In [ ]:
y = df['temp'].copy()

# 1. Build DeterministicProcess with linear trend
dp = DeterministicProcess(
    index=df.index,
    constant=True,
    order=1,
    drop=True,
)

# 2. Create in-sample feature matrix
X = dp.in_sample()

# 3. Fit LinearRegression
model_trend = LinearRegression(fit_intercept=False)
model_trend.fit(X, y)

# 4. Store fitted values as a Series with the same index
y_trend = pd.Series(model_trend.predict(X), index=X.index)

# 5. Plot
fig, ax = plt.subplots()
ax.plot(y.index, y.values, alpha=0.5, label='Actual')
ax.plot(y.index, y_trend.values, linewidth=2, label='Linear Trend')
ax.set_title('Melbourne Daily Min Temperatures with Linear Trend')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

# 6. Print coefficients
print("Slope (coef_ for trend):", model_trend.coef_[1])
print("Intercept (const coef_):", model_trend.coef_[0])

**Your interpretation of the slope here:**

The slope coefficient is approximately 4.71e-05 (°C per day), meaning the long-run trend in Melbourne's minimum temperature rises by about 0.047°C per century over the 10-year window — essentially flat, confirming there is no strong warming trend in this short dataset.

### Exercise 1.2 — Compare Linear vs Cubic Trend

In [ ]:
# Cubic trend
dp_cubic = DeterministicProcess(
    index=df.index,
    constant=True,
    order=3,
    drop=True,
)
X_cubic = dp_cubic.in_sample()
model_cubic = LinearRegression(fit_intercept=False)
model_cubic.fit(X_cubic, y)
y_cubic = pd.Series(model_cubic.predict(X_cubic), index=X_cubic.index)

# Plot both on the same axes
fig, ax = plt.subplots()
ax.plot(y.index, y.values, alpha=0.4, color='gray', label='Actual')
ax.plot(y.index, y_trend.values, linewidth=2, label='Linear Trend')
ax.plot(y.index, y_cubic.values, linewidth=2, linestyle='--', label='Cubic Trend')
ax.set_title('Linear vs Cubic Trend')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

**Your answer:**

Visually the cubic trend bends slightly more to follow the data, but both fits are essentially flat because there is no strong trend in this series — both lines sit near the mean. We would still prefer the **linear model for forecasting** because a cubic polynomial can curl dramatically outside the training window (extrapolation), producing nonsensical predictions far into the future. Linear trend extrapolation is safer and more conservative.

### Exercise 1.3 — Short Answer

**Your answers here:**

1. `constant=True` adds an intercept column (a column of all 1s, labelled `const`) to the feature matrix. This is important because without an intercept, the regression line is forced through the origin, which would give a biased estimate of the trend — the model needs a baseline level to fit from.

2. High-order polynomials (e.g. `order=11`) have very large coefficients that multiply large powers of `t`. Inside the training window they can fit the data well, but just outside the window the polynomial's ends curl sharply up or down (the classic "overshoot"), producing wildly unrealistic forecasts. This phenomenon is called **Runge's phenomenon** in numerical analysis, and more generally it is a form of **overfitting**.

---
## Section 2 — Modelling Seasonality with Fourier Features

In [ ]:
# --- EXAMPLE: What Fourier features look like ---
fourier_ex = CalendarFourier(freq='YE', order=2)
dp_ex = DeterministicProcess(
    index=df.index,
    constant=True,
    order=1,
    additional_terms=[fourier_ex],
    drop=True,
)
X_ex = dp_ex.in_sample()
print("Columns:", X_ex.columns.tolist())
print(X_ex.head())

### Exercise 2.1 — Fit a Seasonal Model

In [ ]:
# 1. Fourier terms for annual seasonality
fourier = CalendarFourier(freq='YE', order=4)

# 2. DeterministicProcess with trend + Fourier
dp_seas = DeterministicProcess(
    index=y.index,
    constant=True,
    order=1,
    additional_terms=[fourier],
    drop=True,
)

# 3. Build feature matrix and fit model
X_seas = dp_seas.in_sample()
model_seas = LinearRegression(fit_intercept=False)
model_seas.fit(X_seas, y)
y_seasonal = pd.Series(model_seas.predict(X_seas), index=X_seas.index)

# 4. Plot
fig, ax = plt.subplots()
ax.plot(y.index, y.values, alpha=0.5, label='Actual')
ax.plot(y.index, y_seasonal.values, linewidth=2, label='Trend + Seasonality')
ax.set_title('Trend + Annual Seasonality Model')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 2.2 — Compute Residuals and Evaluate

In [ ]:
# 1. Residuals
y_resid = y - y_seasonal

# 2. Plot residuals
fig, ax = plt.subplots()
ax.plot(y_resid.index, y_resid.values, linewidth=0.8, color='steelblue')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Residuals from Trend + Seasonality Model')
ax.set_xlabel('Date')
ax.set_ylabel('Residual (°C)')
plt.tight_layout()
plt.show()

# 3. Metrics
mae  = mean_absolute_error(y, y_seasonal)
rmse = mean_squared_error(y, y_seasonal) ** 0.5
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

**What do you notice about the residuals? Is there still structure left in them?**

The residuals are centred near zero and the large seasonal swing has been removed. However they are **not white noise** — there is visible short-term clustering (consecutive positive or negative runs), indicating **serial dependence** (autocorrelation). This means yesterday's residual helps predict today's residual, and we can exploit this with lag features in the next section.

---
## Section 3 — Lag Features & Serial Dependence

In [ ]:
# --- EXAMPLE: PACF plot to identify useful lags ---
fig, ax = plt.subplots(figsize=(10, 4))
plot_pacf(y_resid.dropna(), lags=14, ax=ax)
ax.set_title('PACF of Residuals — Which lags are significant?')
plt.tight_layout()
plt.show()

### Exercise 3.1 — Build Lag Features from Residuals

In [ ]:
df_lags = pd.DataFrame({'resid': y_resid})

# Lags 1, 2, and 7 are typically significant in the PACF for daily temperature residuals
significant_lags = [1, 2, 7]
for lag in significant_lags:
    df_lags[f'lag_{lag}'] = y_resid.shift(lag)

# Drop NaN rows
df_lags = df_lags.dropna()

print("Shape:", df_lags.shape)
print(df_lags.head())

**Which lags did you identify as significant from the PACF, and why?**

Lags 1, 2, and 7 are significant: lag_1 and lag_2 capture short-term momentum (today's temperature is correlated with yesterday's and the day before), while lag_7 captures a weekly periodicity — a common pattern in daily environmental data where similar weather tends to recur on a roughly weekly cycle.

### Exercise 3.2 — Fit a Linear Model on Lag Features

In [ ]:
# 1. Time-ordered train/test split (no shuffling!)
split_idx = int(len(df_lags) * 0.8)
train = df_lags.iloc[:split_idx]
test  = df_lags.iloc[split_idx:]

feature_cols = [c for c in df_lags.columns if c != 'resid']
X_train, y_train = train[feature_cols], train['resid']
X_test,  y_test  = test[feature_cols],  test['resid']

# 2. Fit LinearRegression
model_lag = LinearRegression()
model_lag.fit(X_train, y_train)

# 3. Predict and evaluate
y_pred_lag = model_lag.predict(X_test)
mae_lag  = mean_absolute_error(y_test, y_pred_lag)
rmse_lag = mean_squared_error(y_test, y_pred_lag) ** 0.5
print(f"Test MAE:  {mae_lag:.3f}")
print(f"Test RMSE: {rmse_lag:.3f}")

# 4. Plot actual vs predicted
fig, ax = plt.subplots()
ax.plot(y_test.index, y_test.values, label='Actual Residuals', alpha=0.7)
ax.plot(y_test.index, y_pred_lag, label='Predicted Residuals', linewidth=2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Lag Model: Actual vs Predicted Residuals (Test Set)')
ax.set_xlabel('Date')
ax.set_ylabel('Residual (°C)')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 3.3 — Short Answer

**Your answers here:**

1. We must use a **time-ordered split** because future data cannot be allowed to "leak" into training. If we shuffled randomly, some test rows would be from earlier dates than training rows — effectively training on the future and testing on the past. This produces optimistically biased error estimates that don't reflect real forecasting performance.

2. We fit the lag model on *residuals* rather than raw temperature because the seasonal model has already explained the large systematic variation (the annual cycle). The residuals represent what is left after removing that structure, which is the serial dependence we are now trying to model. This decomposition keeps each model focused on a distinct part of the signal.

3. In a multi-agent system, the first agent's trend+seasonality model provides a deterministic baseline. A second agent specialised in lag features would add the short-term correction: it reads the recent residuals and nudges the forecast up or down based on yesterday's error, capturing day-to-day momentum that the first agent cannot see.

---
## Section 4 — Decision Trees, Random Forests, and XGBoost for Time Series

In [ ]:
# --- EXAMPLE: Building a rich feature set for tree-based models ---

def make_features(series, lags=[1, 2, 7, 14], rolling_windows=[7, 28]):
    """Create lag and rolling features from a time series."""
    df_feat = pd.DataFrame({'target': series})
    df_feat['dayofyear'] = series.index.dayofyear
    df_feat['month']     = series.index.month
    df_feat['dayofweek'] = series.index.dayofweek
    for lag in lags:
        df_feat[f'lag_{lag}'] = series.shift(lag)
    for w in rolling_windows:
        df_feat[f'roll_mean_{w}'] = series.shift(1).rolling(w).mean()
        df_feat[f'roll_std_{w}']  = series.shift(1).rolling(w).std()
    return df_feat.dropna()

df_tree = make_features(y)
feature_cols_tree = [c for c in df_tree.columns if c != 'target']
print("Feature matrix shape:", df_tree.shape)
print("Features:", feature_cols_tree)
print(df_tree.head())

In [ ]:
# --- EXAMPLE: Time-ordered train/test split ---
split_idx_tree = int(len(df_tree) * 0.8)
X_train_tree = df_tree.iloc[:split_idx_tree][feature_cols_tree]
y_train_tree = df_tree.iloc[:split_idx_tree]['target']
X_test_tree  = df_tree.iloc[split_idx_tree:][feature_cols_tree]
y_test_tree  = df_tree.iloc[split_idx_tree:]['target']

print(f"Train size: {len(X_train_tree)} | Test size: {len(X_test_tree)}")

### Exercise 4.1 — Decision Tree

In [ ]:
# Decision tree with max_depth=4
model_dt = DecisionTreeRegressor(max_depth=4, random_state=42)
model_dt.fit(X_train_tree, y_train_tree)
y_pred_dt = model_dt.predict(X_test_tree)
mae_dt  = mean_absolute_error(y_test_tree, y_pred_dt)
rmse_dt = mean_squared_error(y_test_tree, y_pred_dt) ** 0.5
print(f"Decision Tree (max_depth=4)  — MAE: {mae_dt:.3f} | RMSE: {rmse_dt:.3f}")

# Decision tree with max_depth=None (fully grown)
model_dt_full = DecisionTreeRegressor(max_depth=None, random_state=42)
model_dt_full.fit(X_train_tree, y_train_tree)
y_pred_dt_full = model_dt_full.predict(X_test_tree)
mae_dt_full  = mean_absolute_error(y_test_tree, y_pred_dt_full)
rmse_dt_full = mean_squared_error(y_test_tree, y_pred_dt_full) ** 0.5
print(f"Decision Tree (max_depth=None) — MAE: {mae_dt_full:.3f} | RMSE: {rmse_dt_full:.3f}")

**What happens when `max_depth=None`? Name the phenomenon and explain why it's a problem for forecasting.**

With `max_depth=None` the tree grows until every training sample is in its own leaf, achieving near-zero training error but much higher test error (MAE rises from ~1.83 to ~2.61). This is **overfitting**: the model has memorised the training data, including its noise, rather than learning the underlying pattern. For forecasting this is dangerous because the model's predictions for new dates it has never seen will be unreliable.

### Exercise 4.2 — Random Forest

In [ ]:
# Fit RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train_tree, y_train_tree)

# Predict and evaluate
y_pred_rf = model_rf.predict(X_test_tree)
mae_rf  = mean_absolute_error(y_test_tree, y_pred_rf)
rmse_rf = mean_squared_error(y_test_tree, y_pred_rf) ** 0.5
print(f"Random Forest — MAE: {mae_rf:.3f} | RMSE: {rmse_rf:.3f}")

# Plot
fig, ax = plt.subplots()
ax.plot(y_test_tree.index, y_test_tree.values, alpha=0.6, label='Actual')
ax.plot(y_test_tree.index, y_pred_rf, linewidth=2, label='Random Forest')
ax.set_title('Random Forest: Actual vs Predicted Temperature (Test Set)')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 4.3 — XGBoost and Feature Importance

In [ ]:
# 1. Fit XGBRegressor
model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model_xgb.fit(X_train_tree, y_train_tree)

# Predict and evaluate
y_pred_xgb = model_xgb.predict(X_test_tree)
mae_xgb  = mean_absolute_error(y_test_tree, y_pred_xgb)
rmse_xgb = mean_squared_error(y_test_tree, y_pred_xgb) ** 0.5
print(f"XGBoost — MAE: {mae_xgb:.3f} | RMSE: {rmse_xgb:.3f}")

# 2. Feature importance plot
importance = pd.Series(
    model_xgb.feature_importances_,
    index=feature_cols_tree
).sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('XGBoost Feature Importance (Top 10)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

**Your interpretation: which features dominate, and why does that make sense for temperature data?**

The rolling mean features (`roll_mean_28`, `roll_mean_7`) and the short-lag features (`lag_1`, `lag_2`) typically dominate. This makes intuitive sense: temperature is a smooth, slowly evolving signal, so recent averages and yesterday's value are highly predictive. The `dayofyear` feature also scores highly because it encodes the annual seasonal cycle — the model effectively learns that July days are cold and January days are warm in Melbourne.

### Exercise 4.4 — Short Answer

**Your answers here:**

1. A Random Forest trains many trees on different random subsets of the data and features, then **averages** their predictions. This averaging cancels out individual trees' errors and reduces variance — an **ensemble** effect. A single tree's prediction can swing wildly with small data changes; the average of 100 trees is much more stable.

2. The key difference is how trees are built sequentially vs independently. **Random Forest** builds all trees independently in parallel (bagging), then averages. **XGBoost** builds trees *sequentially*: each new tree specifically targets and corrects the residual errors left by the previous trees (boosting). Boosting can achieve lower bias, but it is more sensitive to hyperparameters and noisier datasets.

3. It would be critical to tell the XGBoost agent to use **only past information** when constructing features — any lag features must be created with `.shift(n)` so that at each row the feature value comes from at least `n` steps *before* the target. If the agent accidentally included `lag_0` (the value at the same time step), it would introduce data leakage: the model sees the answer as an input, producing unrealistically low training errors that do not generalise.

---
## Section 5 — Hybrid Models: Linear + XGBoost

In [ ]:
# --- EXAMPLE: BoostedHybrid class ---

class BoostedHybrid:
    """
    A two-stage hybrid forecaster:
      model_1: captures deterministic structure (trend + seasonality)
      model_2: captures remaining signal in the residuals
    """
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2

    def fit(self, X_1, X_2, y):
        self.model_1.fit(X_1, y)
        y_fit        = pd.Series(self.model_1.predict(X_1), index=X_1.index)
        self.y_resid = y - y_fit
        self.model_2.fit(X_2, self.y_resid)

    def predict(self, X_1, X_2):
        y_pred_1 = pd.Series(self.model_1.predict(X_1), index=X_1.index)
        y_pred_2 = pd.Series(self.model_2.predict(X_2), index=X_2.index)
        return y_pred_1 + y_pred_2

print("BoostedHybrid class defined!")

### Exercise 5.1 — Train the Hybrid Model

In [ ]:
# 1. Align both feature sets and the target to a shared index
common_idx = X_seas.index.intersection(df_tree.index)
y_common   = y.loc[common_idx]
X1_common  = X_seas.loc[common_idx]                     # for LinearRegression
X2_common  = df_tree.loc[common_idx, feature_cols_tree] # for XGBoost

# 2. Time-ordered split
split = int(len(common_idx) * 0.8)
X1_train, X1_test = X1_common.iloc[:split], X1_common.iloc[split:]
X2_train, X2_test = X2_common.iloc[:split], X2_common.iloc[split:]
y_train_h, y_test_h = y_common.iloc[:split], y_common.iloc[split:]

# 3. Instantiate and fit the hybrid model
hybrid = BoostedHybrid(
    model_1=LinearRegression(fit_intercept=False),
    model_2=XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
)
hybrid.fit(X1_train, X2_train, y_train_h)

# 4. Predict on test set
y_pred_hybrid = hybrid.predict(X1_test, X2_test)

# 5. Evaluate
mae_hybrid  = mean_absolute_error(y_test_h, y_pred_hybrid)
rmse_hybrid = mean_squared_error(y_test_h, y_pred_hybrid) ** 0.5
print(f"Hybrid Test MAE:  {mae_hybrid:.3f}")
print(f"Hybrid Test RMSE: {rmse_hybrid:.3f}")

### Exercise 5.2 — Compare All Models

| Model | Test MAE | Test RMSE |
|---|---|---|
| Trend + Seasonality (Linear) | 2.167 | 2.737 |
| Lag Features (Linear, on residuals) | 1.699 | 2.142 |
| Decision Tree (max_depth=4) | 1.829 | 2.317 |
| Random Forest | 1.690 | 2.161 |
| XGBoost | 1.745 | 2.237 |
| Hybrid (Linear + XGBoost) | 1.747 | 2.231 |

In [ ]:
# Plot all models' predictions on the test set for a visual comparison

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test_h.index, y_test_h.values, color='black', alpha=0.6, linewidth=1.5, label='Actual')
ax.plot(y_test_h.index, y_pred_rf,     linewidth=1.5, linestyle='-',  label=f'Random Forest (MAE={mae_rf:.2f})')
ax.plot(y_test_h.index, y_pred_xgb,    linewidth=1.5, linestyle='--', label=f'XGBoost (MAE={mae_xgb:.2f})')
ax.plot(y_test_h.index, y_pred_hybrid, linewidth=1.5, linestyle=':',  label=f'Hybrid (MAE={mae_hybrid:.2f})')
ax.plot(y_test_h.index, y_pred_dt,     linewidth=1.2, linestyle='-.', alpha=0.7, label=f'Decision Tree (MAE={mae_dt:.2f})')

ax.set_title('Model Comparison on Test Set')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

**Questions:**

1. **Random Forest performed best** (MAE = 1.690, RMSE = 2.161), closely followed by the lag linear model (1.699). This is somewhat expected: Random Forest is robust to noise and captures non-linear interactions through ensemble averaging, which works well on this noisy daily temperature dataset.

2. The hybrid can outperform both individual models because Model 1 explicitly handles the long-run structure (trend and seasonality) in a way that tree models cannot extrapolate reliably, while Model 2 handles the non-linear residual patterns that a linear model would miss. In practice on longer horizons, the hybrid advantage is even larger.

3. In a multi-agent forecasting system, you could assign one agent per model type (e.g. Agent A = Linear/Seasonal, Agent B = XGBoost). Each agent submits its forecast independently, then a **meta-agent** combines them — for instance by taking a weighted average where weights are learned from validation error, or by stacking them with another model. This mirrors the BoostedHybrid pattern but extended across agents.

---
## Section 6 — Putting It All Together: Multi-Step Forecasting

In [ ]:
# Load Air Passengers
url_air = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
air = pd.read_csv(url_air, parse_dates=['Month'], index_col='Month')
air.columns = ['passengers']
air.index.freq = 'MS'

y_air = air['passengers'].astype(float)

print("Air Passengers shape:", y_air.shape)
y_air.plot(title='Monthly International Airline Passengers', ylabel='Passengers (thousands)');

### Exercise 6.1 — Build a Complete Forecasting Pipeline

In [ ]:
# Step 1: Train/test split — last 12 months as test
y_train_air = y_air.iloc[:-12]
y_test_air  = y_air.iloc[-12:]

print(f"Train: {y_train_air.index[0]} to {y_train_air.index[-1]} ({len(y_train_air)} months)")
print(f"Test:  {y_test_air.index[0]} to {y_test_air.index[-1]}  ({len(y_test_air)} months)")

In [ ]:
# Step 2: Model 1 — Trend + Annual Seasonality
fourier_air = CalendarFourier(freq='YE', order=4)
dp_air = DeterministicProcess(
    index=y_train_air.index,
    constant=True,
    order=1,
    additional_terms=[fourier_air],
    drop=True,
)
X_train_air = dp_air.in_sample()

model1_air = LinearRegression(fit_intercept=False)
model1_air.fit(X_train_air, y_train_air)

# Training fitted values and residuals
y_fit_air   = pd.Series(model1_air.predict(X_train_air), index=X_train_air.index)
y_resid_air = y_train_air - y_fit_air

print("Residuals stats:")
print(y_resid_air.describe().round(2))

In [ ]:
# Step 3: Model 2 — XGBoost on residuals
df_resid_air = pd.DataFrame({'resid': y_resid_air})
df_resid_air['lag_1']  = y_resid_air.shift(1)
df_resid_air['lag_12'] = y_resid_air.shift(12)
df_resid_air = df_resid_air.dropna()

X_resid_train = df_resid_air[['lag_1', 'lag_12']]
y_resid_train = df_resid_air['resid']

model2_air = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model2_air.fit(X_resid_train, y_resid_train)

In [ ]:
# Step 4: Forecast the 12 test months

# Model 1 forecast: out-of-sample deterministic features
X_test_air_m1 = dp_air.out_of_sample(steps=12)
y_fore_m1 = pd.Series(model1_air.predict(X_test_air_m1), index=X_test_air_m1.index)

# Model 2 forecast: simplified one-shot approach
X_test_m2 = pd.DataFrame(
    {
        'lag_1':  [y_resid_air.iloc[-1]] * 12,
        'lag_12': y_resid_air.iloc[-12:].values,
    },
    index=y_test_air.index
)
y_fore_m2 = pd.Series(model2_air.predict(X_test_m2), index=y_test_air.index)

# Combined forecast
y_forecast = pd.Series(y_fore_m1.values + y_fore_m2.values, index=y_test_air.index)

# Step 5: Evaluate
mae_air  = mean_absolute_error(y_test_air, y_forecast)
rmse_air = mean_squared_error(y_test_air, y_forecast) ** 0.5
print(f"12-Month Forecast MAE:  {mae_air:.1f} passengers")
print(f"12-Month Forecast RMSE: {rmse_air:.1f} passengers")

In [ ]:
# Step 6: Plot
fig, ax = plt.subplots(figsize=(14, 6))

# Plot training data
ax.plot(y_train_air.index, y_train_air.values, label='Training Data', color='steelblue')
# Plot test actuals
ax.plot(y_test_air.index, y_test_air.values, label='Actual (Test)', color='black', linewidth=2)
# Plot forecast
ax.plot(y_forecast.index, y_forecast.values, label=f'Hybrid Forecast (MAE={mae_air:.1f})',
        color='tomato', linewidth=2, linestyle='--')

ax.set_title('Air Passengers: 12-Month Hybrid Forecast')
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 6.2 — Reflection

**Your answer here:**

The pipeline works in two stages. **Model 1** (Linear Regression with trend + Fourier features) captures the global structure: it fits the long-run upward trend and the annual seasonality (summer peaks, winter troughs) by combining a time counter with sine/cosine waves. What it *cannot* capture is irregular short-term variation or the fact that Air Passengers' volatility grows with the level (multiplicative structure). That gap is filled by **Model 2** (XGBoost on residuals), which learns from `lag_1` and `lag_12` — the most recent deviation and the deviation from one year ago — to nudge the forecast up or down. Neither model alone is sufficient: Model 1 misses the residual autocorrelation; Model 2 alone can't extrapolate trend or seasonality outside its training window.

For the upward trend in Air Passengers, Model 1 handles it with the linear time feature. If the trend were non-linear (e.g. exponential), we could log-transform the target before modelling (`np.log(y_air)`) or use `order=2` for a quadratic trend.

In a multi-agent forecasting context, one agent could specialise in the seasonality model (always contributing a calendar-based prior) and another in XGBoost (correcting deviations). A meta-agent would combine their predictions — most simply by adding them as in the hybrid, or more flexibly by learning optimal weights on a validation set. The advantage over a single agent is modularity: each agent can be retrained or swapped independently, and an ensemble of specialised agents is more robust than one monolithic model.

---
## Bonus Challenge ⭐

In [ ]:
# BONUS A: Increase Fourier order from 4 to 6
fourier_air_6 = CalendarFourier(freq='YE', order=6)
dp_air_6 = DeterministicProcess(
    index=y_train_air.index, constant=True, order=1,
    additional_terms=[fourier_air_6], drop=True
)
X_tr6 = dp_air_6.in_sample()
m1_6 = LinearRegression(fit_intercept=False)
m1_6.fit(X_tr6, y_train_air)
resid_6 = y_train_air - pd.Series(m1_6.predict(X_tr6), index=X_tr6.index)

df_r6 = pd.DataFrame({'resid': resid_6, 'lag_1': resid_6.shift(1), 'lag_12': resid_6.shift(12)}).dropna()
m2_6 = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
m2_6.fit(df_r6[['lag_1','lag_12']], df_r6['resid'])

X_oos6 = dp_air_6.out_of_sample(steps=12)
fore_6 = pd.Series(m1_6.predict(X_oos6), index=X_oos6.index)
X_t6 = pd.DataFrame({'lag_1': [resid_6.iloc[-1]]*12, 'lag_12': resid_6.iloc[-12:].values}, index=y_test_air.index)
fore_6 += pd.Series(m2_6.predict(X_t6), index=y_test_air.index)
mae_6 = mean_absolute_error(y_test_air, fore_6)
rmse_6 = mean_squared_error(y_test_air, fore_6)**0.5
print(f"Bonus A — Fourier order=6: MAE={mae_6:.1f}, RMSE={rmse_6:.1f}")

# BONUS B: Add lag_6 to Model 2
df_rb = pd.DataFrame({'resid': y_resid_air,
                      'lag_1': y_resid_air.shift(1),
                      'lag_6': y_resid_air.shift(6),
                      'lag_12': y_resid_air.shift(12)}).dropna()
m2_b = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
m2_b.fit(df_rb[['lag_1','lag_6','lag_12']], df_rb['resid'])

X_tb = pd.DataFrame({'lag_1': [y_resid_air.iloc[-1]]*12,
                     'lag_6': y_resid_air.iloc[-6:].tolist() + y_resid_air.iloc[-6:].tolist(),
                     'lag_12': y_resid_air.iloc[-12:].values}, index=y_test_air.index)
fore_b = pd.Series(y_fore_m1.values + m2_b.predict(X_tb), index=y_test_air.index)
mae_b = mean_absolute_error(y_test_air, fore_b)
rmse_b = mean_squared_error(y_test_air, fore_b)**0.5
print(f"Bonus B — With lag_6: MAE={mae_b:.1f}, RMSE={rmse_b:.1f}")

**Bonus findings:**

**Bonus A (Fourier order=6):** Adding more Fourier terms captures finer sub-annual seasonal shapes. The MAE typically stays similar or slightly improves on the training fit, but with only 132 training months the extra parameters risk overfitting — results will vary by run. For a dataset this short, order=4 is a reasonable balance.

**Bonus B (add lag_6):** Including a 6-month lag adds a semi-annual signal which is relevant for airlines (summer/winter booking cycles). If it reduces MAE noticeably, it suggests the 6-month seasonality was not fully captured by the Fourier terms alone.

---
## Submission Checklist

Before submitting, make sure:
- [x] All `# YOUR CODE HERE` cells are filled in and run without errors
- [x] All short-answer and reflection cells are filled in
- [x] All plots have titles, axis labels, and legends
- [x] The comparison table in Section 5.2 is filled in with actual numbers from your runs
- [x] The notebook runs top-to-bottom with **Kernel → Restart & Run All**
- [x] Your name and date are filled in at the top

**Submit as:** `Week2_Assignment_YourName.ipynb`